In [1]:
import torch
from google.colab import drive

drive.mount("/content/drive")

DATA_DIR: str = "/content/drive/MyDrive/high-risk-data"
print(f"GPU: {torch.cuda.is_available()} | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

import os, sys
print(sorted(os.listdir(DATA_DIR)))
sys.path.insert(0, DATA_DIR)

import pandas as pd
from eval_utils import LABEL_COLS
data: pd.DataFrame = pd.read_csv(f"{DATA_DIR}/sbdh_splits.csv")
print(f"rows: {len(data):,} | splits: {data['split'].value_counts().to_dict()} | labels: {len(LABEL_COLS)}")

Mounted at /content/drive
GPU: True | Tesla T4
['eval_utils.py', 'sbdh_splits.csv']
rows: 6,861 | splits: {'train': 4803, 'dev': 1029, 'test': 1029} | labels: 8


In [2]:
from transformers import AutoTokenizer
import numpy as np

MODEL_NAME: str = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

lengths: np.ndarray = np.array([
    len(tokenizer(t, add_special_tokens=True)["input_ids"])
    for t in data["SOCIAL_HISTORY_TEXT"].astype(str)
])

for cap in (128, 256, 512):
    print(f"> {cap} tokens: {(lengths > cap).sum():>4} notes ({(lengths > cap).mean():.1%})")
print(f"median: {int(np.median(lengths))} | p95: {int(np.percentile(lengths, 95))} | max: {lengths.max()}")

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

> 128 tokens:  926 notes (13.5%)
> 256 tokens:  103 notes (1.5%)
> 512 tokens:   14 notes (0.2%)
median: 63 | p95: 176 | max: 1407


In [3]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel

MAX_LEN: int = 256
device = torch.device("cuda")

train_df = data[data["split"] == "train"].reset_index(drop=True)
dev_df = data[data["split"] == "dev"].reset_index(drop=True)
test_df = data[data["split"] == "test"].reset_index(drop=True)

N_CLASSES: dict[str, int] = {col: int(data[col].max()) + 1 for col in LABEL_COLS}


class SBDHDataset(Dataset):
    def __init__(self, df: pd.DataFrame) -> None:
        self.enc = tokenizer(df["SOCIAL_HISTORY_TEXT"].astype(str).tolist(), truncation=True,
                             max_length=MAX_LEN, padding="max_length", return_tensors="pt")
        self.labels = torch.tensor(df[LABEL_COLS].values, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, i: int) -> dict[str, torch.Tensor]:
        return {"input_ids": self.enc["input_ids"][i], "attention_mask": self.enc["attention_mask"][i], "labels": self.labels[i]}


class MultiHeadBERT(nn.Module):
    def __init__(self, model_name: str, n_classes: dict[str, int], dropout: float = 0.1) -> None:
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden: int = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.heads = nn.ModuleDict({col: nn.Linear(hidden, k) for col, k in n_classes.items()})

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> dict[str, torch.Tensor]:
        cls = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        cls = self.dropout(cls)
        return {col: head(cls) for col, head in self.heads.items()}


def class_weights(col: str) -> torch.Tensor:
    counts = np.bincount(train_df[col], minlength=N_CLASSES[col]).astype(float)
    w = np.sqrt(counts.sum() / (len(counts) * np.maximum(counts, 1)))
    return torch.tensor(w / w.mean(), dtype=torch.float, device=device)

loss_fns: dict[str, nn.Module] = {col: nn.CrossEntropyLoss(weight=class_weights(col)) for col in LABEL_COLS}

train_ds, dev_ds, test_ds = SBDHDataset(train_df), SBDHDataset(dev_df), SBDHDataset(test_df)
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)
dev_dl = DataLoader(dev_ds, batch_size=64)
test_dl = DataLoader(test_ds, batch_size=64)

model = MultiHeadBERT(MODEL_NAME, N_CLASSES).to(device)

# sanity check: one forward pass
batch = next(iter(train_dl))
with torch.no_grad():
    out = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
    loss = sum(loss_fns[c](out[c], batch["labels"][:, i].to(device)) for i, c in enumerate(LABEL_COLS))
print(f"N_CLASSES: {N_CLASSES}")
print(f"head shapes: {[tuple(out[c].shape) for c in LABEL_COLS]}")
print(f"initial loss: {loss.item():.3f} | params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print({c: [round(x, 2) for x in class_weights(c).tolist()] for c in ["sdoh_environment", "behavior_drug"]})

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


N_CLASSES: {'sdoh_community_present': 2, 'sdoh_community_absent': 2, 'sdoh_education': 2, 'sdoh_economics': 3, 'sdoh_environment': 3, 'behavior_alcohol': 5, 'behavior_tobacco': 5, 'behavior_drug': 5}
head shapes: [(16, 2), (16, 2), (16, 2), (16, 3), (16, 3), (16, 5), (16, 5), (16, 5)]
initial loss: 10.065 | params: 108.3M
{'sdoh_environment': [0.43, 0.33, 2.24], 'behavior_drug': [0.3, 1.36, 1.3, 0.42, 1.61]}


In [4]:
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
import copy, time

EPOCHS: int = 4
LR: float = 2e-5
CKPT: str = f"{DATA_DIR}/bert_best.pt"

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps: int = len(train_dl) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda")


def predict(dl: DataLoader) -> dict[str, np.ndarray]:
    model.eval()
    preds: dict[str, list[np.ndarray]] = {c: [] for c in LABEL_COLS}
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for b in dl:
            out = model(b["input_ids"].to(device), b["attention_mask"].to(device))
            for c in LABEL_COLS:
                preds[c].append(out[c].argmax(-1).cpu().numpy())
    return {c: np.concatenate(v) for c, v in preds.items()}


def mean_macro_f1(df: pd.DataFrame, preds: dict[str, np.ndarray]) -> tuple[float, dict[str, float]]:
    per: dict[str, float] = {c: f1_score(df[c], preds[c], average="macro", zero_division=0) for c in LABEL_COLS}
    return float(np.mean(list(per.values()))), per


best_f1: float = -1.0
torch.manual_seed(42)
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0, running = time.time(), 0.0
    for b in train_dl:
        optimizer.zero_grad()
        with torch.autocast("cuda", dtype=torch.float16):
            out = model(b["input_ids"].to(device), b["attention_mask"].to(device))
            loss = sum(loss_fns[c](out[c].float(), b["labels"][:, i].to(device)) for i, c in enumerate(LABEL_COLS))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running += loss.item()

    dev_f1, _ = mean_macro_f1(dev_df, predict(dev_dl))
    flag: str = ""
    if dev_f1 > best_f1:
        best_f1 = dev_f1
        torch.save(model.state_dict(), CKPT)
        flag = "  <- best, saved"
    print(f"epoch {epoch} | train loss {running / len(train_dl):.3f} | dev mean macro-F1 {dev_f1:.3f} | {time.time() - t0:.0f}s{flag}")

/tmp/ipykernel_494/3509066797.py:46: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


epoch 1 | train loss 6.851 | dev mean macro-F1 0.684 | 68s  <- best, saved
epoch 2 | train loss 3.435 | dev mean macro-F1 0.829 | 72s  <- best, saved
epoch 3 | train loss 2.300 | dev mean macro-F1 0.852 | 69s  <- best, saved
epoch 4 | train loss 1.849 | dev mean macro-F1 0.856 | 74s  <- best, saved


In [5]:
from sklearn.metrics import classification_report

model.load_state_dict(torch.load(CKPT))
dev_pred_bert: dict[str, np.ndarray] = predict(dev_dl)
_, dev_per_bert = mean_macro_f1(dev_df, dev_pred_bert)

REGEX_DEV: dict[str, float] = {
    "sdoh_community_present": 0.871, "sdoh_community_absent": 0.795, "sdoh_education": 0.830,
    "sdoh_economics": 0.834, "sdoh_environment": 0.765,
    "behavior_alcohol": 0.541, "behavior_tobacco": 0.607, "behavior_drug": 0.522,
}
cmp = pd.DataFrame({"regex": REGEX_DEV, "bert": dev_per_bert})
cmp["gain"] = cmp["bert"] - cmp["regex"]
cmp.loc["MEAN"] = cmp.mean()
print(cmp.round(3).to_string())

# does BERT ever predict unsure
for c in ["behavior_alcohol", "behavior_tobacco", "behavior_drug"]:
    print(f"\n{c}")
    print(classification_report(dev_df[c], dev_pred_bert[c], zero_division=0, digits=2))

                        regex   bert   gain
sdoh_community_present  0.871  0.956  0.085
sdoh_community_absent   0.795  0.919  0.124
sdoh_education          0.830  0.890  0.060
sdoh_economics          0.834  0.904  0.070
sdoh_environment        0.765  0.802  0.037
behavior_alcohol        0.541  0.787  0.246
behavior_tobacco        0.607  0.860  0.253
behavior_drug           0.522  0.726  0.204
MEAN                    0.721  0.856  0.135

behavior_alcohol
              precision    recall  f1-score   support

           0       0.97      0.88      0.92       267
           1       0.87      0.89      0.88       288
           2       0.57      0.57      0.57        56
           3       0.92      0.96      0.94       374
           4       0.60      0.66      0.63        44

    accuracy                           0.88      1029
   macro avg       0.79      0.79      0.79      1029
weighted avg       0.89      0.88      0.88      1029


behavior_tobacco
              precision    recall  

In [6]:
test_pred_bert: dict[str, np.ndarray] = predict(test_dl)
test_mean, test_per_bert = mean_macro_f1(test_df, test_pred_bert)

REGEX_TEST: dict[str, float] = {
    "sdoh_community_present": 0.884, "sdoh_community_absent": 0.785, "sdoh_education": 0.834,
    "sdoh_economics": 0.811, "sdoh_environment": 0.848,
    "behavior_alcohol": 0.580, "behavior_tobacco": 0.621, "behavior_drug": 0.529,
}
test_cmp = pd.DataFrame({"regex": REGEX_TEST, "bert": test_per_bert})
test_cmp["gain"] = test_cmp["bert"] - test_cmp["regex"]
test_cmp.loc["MEAN"] = test_cmp.mean()
print(test_cmp.round(3).to_string())

# save to Drive: metrics
test_cmp.round(4).to_csv(f"{DATA_DIR}/test_scores_bert_vs_regex.csv")
pd.DataFrame({"ROW_ID": test_df["ROW_ID"], **{c: test_pred_bert[c] for c in LABEL_COLS}}).to_csv(
    f"{DATA_DIR}/test_pred_bert.csv", index=False)
pd.DataFrame({"ROW_ID": dev_df["ROW_ID"], **{c: dev_pred_bert[c] for c in LABEL_COLS}}).to_csv(
    f"{DATA_DIR}/dev_pred_bert.csv", index=False)
print("saved: test_scores_bert_vs_regex.csv, test_pred_bert.csv, dev_pred_bert.csv")

                        regex   bert   gain
sdoh_community_present  0.884  0.962  0.078
sdoh_community_absent   0.785  0.897  0.112
sdoh_education          0.834  0.913  0.079
sdoh_economics          0.811  0.897  0.086
sdoh_environment        0.848  0.868  0.020
behavior_alcohol        0.580  0.813  0.233
behavior_tobacco        0.621  0.848  0.227
behavior_drug           0.529  0.704  0.175
MEAN                    0.736  0.863  0.126
saved: test_scores_bert_vs_regex.csv, test_pred_bert.csv, dev_pred_bert.csv
